## Project 2: Supervised Learning (Fraud Detection Pipeline)

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score,recall_score,roc_auc_score,classification_report
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
df=pd.read_csv('clean_feature_engineered_orders.csv')
X=df.drop('Fraud',axis=1);y=df['Fraud']

In [5]:
cat=X.select_dtypes(include='object').columns
num=X.select_dtypes(exclude='object').columns
pre=ColumnTransformer([
('num',Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),num),
('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)
])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

In [6]:
lr=ImbPipeline([('pre',pre),('smote',SMOTE(random_state=42)),('clf',LogisticRegression(max_iter=1000))])
lr.fit(X_train,y_train)
pred=lr.predict(X_test);prob=lr.predict_proba(X_test)[:,1]
print('LR Precision',precision_score(y_test,pred))
print('LR Recall',recall_score(y_test,pred))
print('LR ROC-AUC',roc_auc_score(y_test,prob))
print(classification_report(y_test,pred))

LR Precision 0.8
LR Recall 1.0
LR ROC-AUC 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       232
           1       0.80      1.00      0.89         4

    accuracy                           1.00       236
   macro avg       0.90      1.00      0.94       236
weighted avg       1.00      1.00      1.00       236



In [7]:
rf=ImbPipeline([('pre',pre),('smote',SMOTE(random_state=42)),('clf',RandomForestClassifier(random_state=42))])
grid=GridSearchCV(rf,{
'clf__n_estimators':[100,200],
'clf__max_depth':[10,None]
},scoring='roc_auc',cv=3)
grid.fit(X_train,y_train)
best=grid.best_estimator_
pred=best.predict(X_test);prob=best.predict_proba(X_test)[:,1]
print(grid.best_params_)
print('RF Precision',precision_score(y_test,pred))
print('RF Recall',recall_score(y_test,pred))
print('RF ROC-AUC',roc_auc_score(y_test,prob))
print(classification_report(y_test,pred))

{'clf__max_depth': None, 'clf__n_estimators': 100}
RF Precision 0.0
RF Recall 0.0
RF ROC-AUC 1.0
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       232
           1       0.00      0.00      0.00         4

    accuracy                           0.98       236
   macro avg       0.49      0.50      0.50       236
weighted avg       0.97      0.98      0.97       236



C:\Users\Neelesh Ranpuriya\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Neelesh Ranpuriya\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Neelesh Ranpuriya\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-pa